In [1]:
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np
from ast import literal_eval
import re

In [2]:
def rapi(s):
    ret = ''
    for ch in str(s):
        if ch == '-' or ch.isalpha() or ch == ' ':
            ret += ch
    return ret.strip().lower()

In [3]:
def ubah2(s):
    s = str(s)
    while len(s) > 0 and (not s[0].isalpha()):
        s = s[1:]
    while len(s) > 0 and (not s[-1].isalpha()):
        s = s[:-1]
    return s

In [4]:
def ubah(ls):
    while len(ls) > 0:
        while((len(ls[0][0]) > 0) and (not str(ls[0][0])[0].isalpha())):
            ls[0][0] = ls[0][0][1:]
        if len(ls[0][0]) == 0:
            ls = ls[1:]
        else:
            break
    while len(ls) > 0:
        while((len(ls[-1][0]) > 0) and (not str(ls[-1][0])[-1].isalpha())):
            ls[-1][0] = ls[-1][0][:-1]
        if len(ls[-1][0]) == 0:
            ls = ls[:-1]
        else:
            break
    return ls

In [5]:
def tidyUp(df, arrow_list):
    last_main_entry = ''
    kata_asal_list = []
    POS_list = []
    kata_tujuan_list = []
    makna_list = []
    contoh_kalimat_list = []
    contoh_kalimat_font_size_pos_list = []
    main_entry_list = []
    kata_turunan_list = []
    is_anomali_list = []
    need_update_symbol = 0
    main_entry_symbol = ''
    subentry_symbol = ''
    data_simbol_df = data_simbol[data_simbol['nomor'] == int(nomor)]
    if len(data_simbol_df) > 0:
        need_update_symbol = 1
        for i, row in data_simbol_df.iterrows():
            main_entry_symbol = row['entri_pokok']
            subentry_symbol = row['subentri']
    print(main_entry_symbol, subentry_symbol)

    for i, _row in df.iterrows():
        row = df.loc[i].copy()
        ada = 0
        if row['is_anomali']:
            for arrow in arrow_list:
                split_list = str(row['kata_asal']).split(arrow)
                if len(split_list) == 2:
                    new_df = df[df.apply(lambda row: rapi(str(row['kata_asal'])) == rapi(split_list[1]), axis=1)]
                    if len(new_df) == 1:
                        new_row = new_df.iloc[0]
                        kata_asal_tmp = split_list[0]
                        df.loc[i] = new_row
                        df.loc[i,'kata_asal'] = kata_asal_tmp
                        row = new_row.copy()
                        row['kata_asal'] = kata_asal_tmp
                        ada = 1
                        break
        kata_asal = row['kata_asal']
        if row['main_entry'] == 0 and len(str(kata_asal)) > 0:
            if str(kata_asal)[0] == main_entry_symbol or str(kata_asal)[-1] == main_entry_symbol:
                kata_asal = str(kata_asal).replace(main_entry_symbol, str(last_main_entry))

        kata_asal = rapi(kata_asal)
        kata_asal_list.append(kata_asal)

        if row['main_entry'] == 1:
            last_main_entry = kata_asal

        if(pd.isnull(row['POS'])):
            POS_list.append('')
        else:
            POS_list.append(row['POS'])

        new_kata_tujuan = []
        for kata in eval(df.iloc[i]['kata_tujuan']):
            new_kata_tujuan.append(rapi(kata))
        kata_tujuan_list.append(new_kata_tujuan)

        new_makna = []
        for kata in eval(df.iloc[i]['makna']):
            new_makna.append(rapi(kata))
        makna_list.append(new_makna)

        if(pd.isnull(row['contoh_kalimat'])):
            contoh_kalimat = ''
        else:
            contoh_kalimat = row['contoh_kalimat']

        if need_update_symbol:
            contoh_kalimat = str(contoh_kalimat).replace(main_entry_symbol, last_main_entry)
            if row['main_entry'] == 0:
                contoh_kalimat = str(contoh_kalimat).replace(subentry_symbol, str(kata_asal))

        contoh_kalimat_list.append(ubah2(str(contoh_kalimat)))

        new_contoh_kalimat_font_size_pos = []
        for contoh_kalimat in eval(row['contoh_kalimat_font_size_pos']):
            if need_update_symbol:
                awal = contoh_kalimat[0]
                contoh_kalimat[0] = str(contoh_kalimat[0]).replace(main_entry_symbol, last_main_entry)
                if row['main_entry'] == 0:
                    contoh_kalimat[0] = str(contoh_kalimat[0]).replace(subentry_symbol, str(kata_asal))
            new_contoh_kalimat_font_size_pos.append(contoh_kalimat)

        contoh_kalimat_font_size_pos_list.append(ubah(new_contoh_kalimat_font_size_pos))

        main_entry_list.append(row['main_entry'])

        if row['main_entry'] == 0 and ('-' not in str(row['kata_asal']) and (last_main_entry + ' ') not in str(row['kata_asal'])) and ((' ' + last_main_entry) not in str(row['kata_asal'])):
            kata_turunan_list.append(1)
        else:
            kata_turunan_list.append(0)

        is_anomali_list.append(row['is_anomali'])


    ret = pd.DataFrame()
    ret['kata_asal'] = kata_asal_list
    ret['POS'] = POS_list
    ret['kata_tujuan'] = kata_tujuan_list
    ret['makna'] = makna_list
    ret['contoh_kalimat'] = contoh_kalimat_list
    ret['contoh_kalimat_font_size_pos'] = contoh_kalimat_font_size_pos_list
    ret['main_entry'] = main_entry_list
    ret['kata_turunan'] = kata_turunan_list
    ret['is_anomali'] = is_anomali_list

    return ret

In [6]:
data_simbol = pd.read_csv('../Ekstraksi/X. Data Pendukung/Penggantian Simbol.csv')

In [7]:
data_simbol

,nomor,entri_pokok,subentri
0,18,--,--
1,54,-,==
2,68,--,~
3,71,--,~


In [8]:
import os
from pathlib import Path
directory = "../Ekstraksi/"
path = directory + '8. Cleaning Informasi Sumber Daya NLP/'
cnt = 0
daftar_kamus = [18,68,71,54,89]
for filename in os.listdir(directory + '7. Ekstraksi Informasi Sumber Daya NLP/'):
    if(filename[-3:] == 'csv' and filename[0].isnumeric()):
        nomor = filename.split('_')[0]
        # if int(nomor) not in daftar_kamus:
            # continue
        f = os.path.join(directory + '7. Ekstraksi Informasi Sumber Daya NLP/', filename)
        df = pd.read_csv(f)
        df = pd.read_csv(f)
        df_nice = tidyUp(df, ['→', '->', '-.', '-+'])
        df_nice.to_csv(path + nomor + "_Nice.csv", index = False)
        cnt += 1
        print('Done Kamus', nomor, '(Total', cnt, 'Kamus)')


 
Done Kamus 10 (Total 1 Kamus)
 
Done Kamus 11 (Total 2 Kamus)
 
Done Kamus 12 (Total 3 Kamus)
 
Done Kamus 13 (Total 4 Kamus)
 
Done Kamus 14 (Total 5 Kamus)
 
Done Kamus 15 (Total 6 Kamus)
 
Done Kamus 16 (Total 7 Kamus)
 
Done Kamus 17 (Total 8 Kamus)
-- --
Done Kamus 18 (Total 9 Kamus)
 
Done Kamus 19 (Total 10 Kamus)
 
Done Kamus 1 (Total 11 Kamus)
 
Done Kamus 20 (Total 12 Kamus)
 
Done Kamus 21 (Total 13 Kamus)
 
Done Kamus 22 (Total 14 Kamus)
 
Done Kamus 23 (Total 15 Kamus)
 
Done Kamus 24 (Total 16 Kamus)
 
Done Kamus 25 (Total 17 Kamus)
 
Done Kamus 26 (Total 18 Kamus)
 
Done Kamus 27 (Total 19 Kamus)
 
Done Kamus 28 (Total 20 Kamus)
 
Done Kamus 29 (Total 21 Kamus)
 
Done Kamus 2 (Total 22 Kamus)
 
Done Kamus 31 (Total 23 Kamus)
 
Done Kamus 32 (Total 24 Kamus)
 
Done Kamus 33 (Total 25 Kamus)
 
Done Kamus 34 (Total 26 Kamus)
 
Done Kamus 35 (Total 27 Kamus)
 
Done Kamus 36 (Total 28 Kamus)
 
Done Kamus 37 (Total 29 Kamus)
 
Done Kamus 38 (Total 30 Kamus)
 
Done Kamus 3 (T